In [3]:
print()

In [4]:

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import numpy as np
import pandas as pd
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
import re
from typing import List

[nltk_data] Downloading package punkt to C:\Users\Prashant
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Prashant
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
def preprocess_caption(caption) -> List[str]:
    """Clean and tokenize a caption"""
    caption = caption.lower()
    caption = re.sub(r'[^a-z0-9\s\']', '', caption)
    tokens = nltk.tokenize.word_tokenize(caption)
    tokens = [token for token in tokens if token.strip() and token not in ["'", '"', "''", "``"]]
    return tokens

def count_tokens(text):
    return len(nltk.word_tokenize(text.lower()))

In [6]:
def apply_augmentations(image_tensor):
    """
    Generate 2 augmented versions of an image:
    1. Original
    2. Horizontal flip
   
    """
    augmented_images = []
    
    # 1. Original
    augmented_images.append(image_tensor)
    
    # 2. Horizontal flip
    augmented_images.append(transforms.functional.hflip(image_tensor))
    
    return augmented_images

In [7]:
print("\nLoading captions...")
captions_df = pd.read_csv('captions.txt')

# Get both shortest and longest caption per image
image_data = []
for image_name in captions_df['image'].unique():
    image_captions = captions_df[captions_df['image'] == image_name]['caption'].tolist()
    shortest = min(image_captions, key=count_tokens)
    longest = max(image_captions, key=count_tokens)
    image_data.append({
        'image': image_name,
        'short_caption': shortest,
        'long_caption': longest
    })

clean_df = pd.DataFrame(image_data)
print(f"Found {len(clean_df)} unique images")


Loading captions...
Found 8091 unique images


In [8]:
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


In [10]:
print("\nCreating augmented dataset...")
all_data = []

for idx, row in clean_df.iterrows():
    img_path = os.path.join('images', row['image'])
    img = Image.open(img_path).convert('RGB')
    
    # Convert to tensor
    img_tensor = base_transform(img)
    
    # Generate 2 augmented versions
    augmented_versions = apply_augmentations(img_tensor)
    
    # Store each version with its captions (raw text, no embeddings!)
    for aug_img in augmented_versions:
        all_data.append({
            'image': aug_img,
            'short_caption': row['short_caption'],
            'long_caption': row['long_caption'],
            'short_token_count': count_tokens(row['short_caption']),
            'long_token_count': count_tokens(row['long_caption'])
        })
    
    if (idx + 1) % 500 == 0:
        print(f"Processed {idx + 1}/{len(clean_df)} images ({(idx + 1) * 2} total samples)")

print(f"\nTotal augmented samples: {len(all_data)}")
print(f"Original images: {len(clean_df)}")
print(f"Augmentation factor: 2x")


Creating augmented dataset...
Processed 500/8091 images (1000 total samples)
Processed 1000/8091 images (2000 total samples)
Processed 1500/8091 images (3000 total samples)
Processed 2000/8091 images (4000 total samples)
Processed 2500/8091 images (5000 total samples)
Processed 3000/8091 images (6000 total samples)
Processed 3500/8091 images (7000 total samples)
Processed 4000/8091 images (8000 total samples)
Processed 4500/8091 images (9000 total samples)
Processed 5000/8091 images (10000 total samples)
Processed 5500/8091 images (11000 total samples)
Processed 6000/8091 images (12000 total samples)
Processed 6500/8091 images (13000 total samples)
Processed 7000/8091 images (14000 total samples)
Processed 7500/8091 images (15000 total samples)
Processed 8000/8091 images (16000 total samples)

Total augmented samples: 16182
Original images: 8091
Augmentation factor: 2x


In [11]:
images_tensor = torch.stack([item['image'] for item in all_data])
short_captions = [item['short_caption'] for item in all_data]
long_captions = [item['long_caption'] for item in all_data]
short_token_counts = [item['short_token_count'] for item in all_data]
long_token_counts = [item['long_token_count'] for item in all_data]

print(f"\nImages tensor shape: {images_tensor.shape}")
print(f"Number of short captions: {len(short_captions)}")
print(f"Number of long captions: {len(long_captions)}")


Images tensor shape: torch.Size([16182, 3, 224, 224])
Number of short captions: 16182
Number of long captions: 16182


In [12]:
class ImageCaptionDataset(Dataset):
    def __init__(self, images, short_captions, long_captions, short_token_counts, long_token_counts):
        self.images = images
        self.short_captions = short_captions
        self.long_captions = long_captions
        self.short_token_counts = short_token_counts
        self.long_token_counts = long_token_counts

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return {
            'image': self.images[idx],
            'short_caption': self.short_captions[idx],
            'long_caption': self.long_captions[idx],
            'short_token_count': self.short_token_counts[idx],
            'long_token_count': self.long_token_counts[idx]
        }

# Create full dataset
full_dataset = ImageCaptionDataset(
    images_tensor, 
    short_captions, 
    long_captions, 
    short_token_counts, 
    long_token_counts
)

In [13]:
np.random.seed(42)
num_samples = len(full_dataset)
indices = np.random.permutation(num_samples)
train_split = int(0.8 * num_samples)
val_split = int(0.9 * num_samples)

train_indices = indices[:train_split]
val_indices = indices[train_split:val_split]
test_indices = indices[val_split:]

print(f"\nDataset splits:")
print(f"Train: {len(train_indices)} samples")
print(f"Val: {len(val_indices)} samples")
print(f"Test: {len(test_indices)} samples")


Dataset splits:
Train: 12945 samples
Val: 1618 samples
Test: 1619 samples


In [15]:
torch.save({
    'images': images_tensor,
    'short_captions': short_captions,
    'long_captions': long_captions,
    'short_token_counts': short_token_counts,
    'long_token_counts': long_token_counts,
    'train_indices': train_indices,
    'val_indices': val_indices,
    'test_indices': test_indices,
    'metadata': {
        'num_original_images': len(clean_df),
        'num_augmented_samples': len(all_data),
        'augmentation_factor': 2,
        'image_shape': (3, 224, 224),
        'augmentations': ['original', 'horizontal_flip'],
        'contains_embeddings': False,  # Important flag!
        'note': 'Embeddings will be generated on-the-fly during training using Word2Vec'
    }
}, 'flickr8k_dataset_raw.pt')

print(f"\n✅ Saved dataset to 'flickr8k_dataset_raw.pt'")
print(f"File size: {os.path.getsize('flickr8k_dataset_raw.pt') / 1e6:.1f} MB")


✅ Saved dataset to 'flickr8k_dataset_raw.pt'
File size: 9744.7 MB
